# 05. Study 1 -- Formula vs. ML Comparison

Read-only analysis on top of the existing pipeline: does the ML blend
actually beat simple clinical formulas (FIB-4, APRI)? Does not touch
`outputs/improved_submission.csv`, `scripts/train.py`, or any saved model.

This notebook produces three CSVs in `outputs/`, each with a distinct role:

1. **`outputs/study1_cindex_comparison.csv`** -- the primary evidence. Every
   score in this file comes from proper cross-validation (formulas) or
   honest out-of-fold prediction (the ML blend), so it is a fair,
   unbiased comparison. This is the file to cite for "does ML beat FIB-4/
   APRI" claims.
2. **`outputs/ranking_comparison_fib4.csv`** and
   **`outputs/ranking_comparison_apri.csv`** -- illustrative, not the
   headline metric. `ml_score` here is the same honest out-of-fold blend
   prediction used for the bootstrap (single 5-fold split, not the
   repeated 3x CV behind Table 1), used to spot individual patients where
   the ML ranking and the formula ranking disagree sharply, for
   discussion-chapter examples. Cite Table 1's mean/std for overall
   performance claims, not these.


In [ ]:
import sys
from pathlib import Path


def _find_repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "liverrisk" / "features.py").exists():
            return p
    raise RuntimeError("Could not locate repo root (liverrisk/features.py not found)")


REPO_ROOT = _find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print("REPO_ROOT:", REPO_ROOT)


In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold

from liverrisk import config
from liverrisk.blend import blend_predictions
from liverrisk.clinical_scores import compute_fib4_apri
from liverrisk.cv import bootstrap_cindex_diff, cv_cindex_blend, cv_cindex_formula
from liverrisk.features import load_features
from liverrisk.models import (
    HAS_XGB,
    RANDOM_STATE,
    make_coxnet_pipeline,
    make_rsf_pipeline,
    make_xgb_pipeline,
    signed_time_label,
)

PROCESSED_DIR = REPO_ROOT / "liverrisk" / "data" / "processed"
RAW_DIR = REPO_ROOT / "liverrisk" / "data" / "raw"
OUTPUTS_DIR = REPO_ROOT / "outputs"
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

N_REPEATS = 3  # trustworthy, not the fast search n_repeats=1 used during grid search
N_BOOT = 1000

X_hep, y_hep, hep_event, hep_time = load_features("hep", PROCESSED_DIR)
X_death, y_death, death_event, death_time = load_features("death", PROCESSED_DIR)

print(f"hepatic: n={len(X_hep)}, events={hep_event.sum()} ({hep_event.mean():.3%})")
print(f"death  : n={len(X_death)}, events={death_event.sum()} ({death_event.mean():.3%})")


## Compute FIB-4 / APRI for both endpoints

In [ ]:
fib4_apri_hep = compute_fib4_apri(X_hep, "hep")
fib4_apri_death = compute_fib4_apri(X_death, "death")

fib4_apri_hep.describe()


## Formula CV scores (FIB-4, APRI) -- honest, cross-validated

`cv_cindex_formula` uses the same `RepeatedStratifiedKFold(random_state=
RANDOM_STATE)` split as the ML models' own CV, at `n_repeats=3` -- a
trustworthy estimate, not the fast `n_repeats=1` used during hyperparameter
search.

In [ ]:
hep_fib4_cv = cv_cindex_formula(X_hep, y_hep, hep_event, fib4_apri_hep["fib4_score"].to_numpy(), n_repeats=N_REPEATS)
death_fib4_cv = cv_cindex_formula(X_death, y_death, death_event, fib4_apri_death["fib4_score"].to_numpy(), n_repeats=N_REPEATS)
hep_apri_cv = cv_cindex_formula(X_hep, y_hep, hep_event, fib4_apri_hep["apri_score"].to_numpy(), n_repeats=N_REPEATS)
death_apri_cv = cv_cindex_formula(X_death, y_death, death_event, fib4_apri_death["apri_score"].to_numpy(), n_repeats=N_REPEATS)

print(f"FIB-4 hepatic: mean={hep_fib4_cv[0]:.4f}, std={hep_fib4_cv[1]:.4f}")
print(f"FIB-4 death  : mean={death_fib4_cv[0]:.4f}, std={death_fib4_cv[1]:.4f}")
print(f"APRI  hepatic: mean={hep_apri_cv[0]:.4f}, std={hep_apri_cv[1]:.4f}")
print(f"APRI  death  : mean={death_apri_cv[0]:.4f}, std={death_apri_cv[1]:.4f}")


## Final blend CV score -- same procedure, `n_repeats=3`

Reuses `cv_cindex_blend` with the tuned `config.blend_weights_hep()`/
`_death()` and `config.xgb_hyperparams_hep()`/`_death()` -- the exact same
blend that produced `outputs/improved_submission.csv`, just cross-validated
here rather than fit once on all the data.

In [ ]:
hep_blend_cv = cv_cindex_blend(
    X_hep, y_hep, hep_event, hep_time,
    n_repeats=N_REPEATS,
    weights=config.blend_weights_hep(),
    xgb_params=config.xgb_hyperparams_hep(),
)
death_blend_cv = cv_cindex_blend(
    X_death, y_death, death_event, death_time,
    n_repeats=N_REPEATS,
    weights=config.blend_weights_death(),
    xgb_params=config.xgb_hyperparams_death(),
)

print(f"Blend hepatic: mean={hep_blend_cv[0]:.4f}, std={hep_blend_cv[1]:.4f}")
print(f"Blend death  : mean={death_blend_cv[0]:.4f}, std={death_blend_cv[1]:.4f}")


## Out-of-fold blend predictions, for the bootstrap

`bootstrap_cindex_diff` needs one fixed score per patient for each method,
not a (mean, std) pair -- FIB-4/APRI already have that (the formula doesn't
fit anything, so `fib4_score`/`apri_score` are unbiased as-is). The blend
does fit per fold, so a single out-of-fold pass (5-fold, same
`RANDOM_STATE` as the rest of the pipeline) gives each patient a blend
score from a model that never saw them -- honest, and directly comparable
to the formula scores. This is a single split (not repeated), used only to
build per-patient arrays for the bootstrap; the trustworthy mean/std above
already comes from the repeated CV in the previous cell.

In [ ]:
def oof_blend_predictions(X, y, event, time, weights, xgb_params, n_splits=5, rsf_n_estimators=150):
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
    oof = np.full(len(X), np.nan)
    y_signed = signed_time_label(event, time) if HAS_XGB else None

    for tr, va in cv.split(X, event.astype(int)):
        X_tr, X_va = X.iloc[tr], X.iloc[va]
        preds = []

        cox = make_coxnet_pipeline(X_tr)
        cox.fit(X_tr, y[tr])
        preds.append(cox.predict(X_va))

        rsf = make_rsf_pipeline(X_tr, n_estimators=rsf_n_estimators)
        rsf.fit(X_tr, y[tr])
        preds.append(rsf.predict(X_va))

        if HAS_XGB:
            xgb = make_xgb_pipeline(X_tr, **xgb_params)
            xgb.fit(X_tr, y_signed[tr])
            preds.append(xgb.predict(X_va))

        oof[va] = blend_predictions(preds, weights)

    assert not np.isnan(oof).any(), "every patient should land in exactly one validation fold"
    return oof


hep_blend_oof = oof_blend_predictions(
    X_hep, y_hep, hep_event, hep_time,
    weights=config.blend_weights_hep(), xgb_params=config.xgb_hyperparams_hep(),
)
death_blend_oof = oof_blend_predictions(
    X_death, y_death, death_event, death_time,
    weights=config.blend_weights_death(), xgb_params=config.xgb_hyperparams_death(),
)
print("OOF blend predictions computed for both endpoints.")


## Bootstrap the C-index gap (blend vs. each formula)

1000 patient-resamples with replacement, each scoring both the OOF blend
and the formula, recording (blend - formula). If the resulting 95%
interval excludes zero, the gap is unlikely to be sampling noise.

FIB-4/APRI are undefined (NaN) for patients missing AST/ALT/platelets
(~34% of the hepatic cohort and ~29% of the death cohort, per the NaN
counts printed above) -- `bootstrap_cindex_diff` handles this itself: for
each resample it drops indices where either score is NaN, using one
shared mask so both scores are evaluated on the exact same patients, and
skips (uncounted) any resample left with fewer than 2 patients or zero
events.

In [ ]:
hep_diff_fib4, hep_fib4_lo, hep_fib4_hi = bootstrap_cindex_diff(
    y_hep, hep_event, hep_time, hep_blend_oof, fib4_apri_hep["fib4_score"].to_numpy(), n_boot=N_BOOT,
)
death_diff_fib4, death_fib4_lo, death_fib4_hi = bootstrap_cindex_diff(
    y_death, death_event, death_time, death_blend_oof, fib4_apri_death["fib4_score"].to_numpy(), n_boot=N_BOOT,
)
hep_diff_apri, hep_apri_lo, hep_apri_hi = bootstrap_cindex_diff(
    y_hep, hep_event, hep_time, hep_blend_oof, fib4_apri_hep["apri_score"].to_numpy(), n_boot=N_BOOT,
)
death_diff_apri, death_apri_lo, death_apri_hi = bootstrap_cindex_diff(
    y_death, death_event, death_time, death_blend_oof, fib4_apri_death["apri_score"].to_numpy(), n_boot=N_BOOT,
)

print(f"Hepatic blend-vs-FIB4 95% CI: [{hep_fib4_lo:.4f}, {hep_fib4_hi:.4f}]")
print(f"Death   blend-vs-FIB4 95% CI: [{death_fib4_lo:.4f}, {death_fib4_hi:.4f}]")
print(f"Hepatic blend-vs-APRI 95% CI: [{hep_apri_lo:.4f}, {hep_apri_hi:.4f}]")
print(f"Death   blend-vs-APRI 95% CI: [{death_apri_lo:.4f}, {death_apri_hi:.4f}]")


## Write `outputs/study1_cindex_comparison.csv`

This is the primary evidence for whether the ML blend outperforms
FIB-4/APRI. Every score here comes from proper cross-validation on the
training cohort -- each patient's score was produced by a model that
never saw that patient during fitting, so this is an honest, unbiased
comparison. The `ci_diff_vs_blend` columns show whether the gap is
statistically real (interval excludes zero) or could be noise.

In [ ]:
cindex_comparison = pd.DataFrame([
    {"method": "FIB-4", "endpoint": "hepatic", "mean_cindex": hep_fib4_cv[0], "std_cindex": hep_fib4_cv[1],
     "ci_diff_vs_blend_lower": hep_fib4_lo, "ci_diff_vs_blend_upper": hep_fib4_hi},
    {"method": "APRI", "endpoint": "hepatic", "mean_cindex": hep_apri_cv[0], "std_cindex": hep_apri_cv[1],
     "ci_diff_vs_blend_lower": hep_apri_lo, "ci_diff_vs_blend_upper": hep_apri_hi},
    {"method": "Blend", "endpoint": "hepatic", "mean_cindex": hep_blend_cv[0], "std_cindex": hep_blend_cv[1],
     "ci_diff_vs_blend_lower": np.nan, "ci_diff_vs_blend_upper": np.nan},
    {"method": "FIB-4", "endpoint": "death", "mean_cindex": death_fib4_cv[0], "std_cindex": death_fib4_cv[1],
     "ci_diff_vs_blend_lower": death_fib4_lo, "ci_diff_vs_blend_upper": death_fib4_hi},
    {"method": "APRI", "endpoint": "death", "mean_cindex": death_apri_cv[0], "std_cindex": death_apri_cv[1],
     "ci_diff_vs_blend_lower": death_apri_lo, "ci_diff_vs_blend_upper": death_apri_hi},
    {"method": "Blend", "endpoint": "death", "mean_cindex": death_blend_cv[0], "std_cindex": death_blend_cv[1],
     "ci_diff_vs_blend_lower": np.nan, "ci_diff_vs_blend_upper": np.nan},
])

cindex_comparison_path = OUTPUTS_DIR / "study1_cindex_comparison.csv"
cindex_comparison.to_csv(cindex_comparison_path, index=False)
print(f"Wrote {cindex_comparison_path} ({len(cindex_comparison)} rows)")
cindex_comparison


## Ranking-disagreement tables

Purpose: find specific patients where the ML ranking and the formula
ranking disagree sharply (large `rank_gap`), then check the real
`event`/`time` columns to see which ranking was actually right for that
patient -- useful for discussion-chapter examples. All patients here are
from the training cohort, not the test set, specifically because the
test set has no known outcomes to check against.

`ml_score` here is the out-of-fold blend prediction computed above for
the bootstrap (`hep_blend_oof`/`death_blend_oof`) -- the same honest,
never-saw-this-patient scores used for the bootstrap CI, not the
production models' optimistic scores on their own training data. Note
this is still a single (non-repeated) 5-fold split, chosen for
simplicity, distinct from the repeated 3x CV behind Table 1's headline
mean/std.

`ml_rank`/`formula_rank` are integer ranks within the endpoint's cohort,
rank 1 being the single highest-risk patient by that scoring method
(ties broken by `method="min"`). `rank_gap = formula_rank - ml_rank`:
**positive `rank_gap` means the ML model ranks this patient as more
urgent (lower rank number) than the formula does; negative means the
formula flags them as more urgent than the ML model does.** Patients
missing labs get a NaN `formula_score`/`formula_rank`/`rank_gap` (row
kept, not dropped) -- FIB-4/APRI simply can't rank them. The written CSVs
are sorted by `rank_gap.abs()` descending, so the biggest ML-vs-formula
disagreements come first.

In [ ]:
train_df = pd.read_csv(RAW_DIR / "train.csv")
id_col = "patient_id_anon" if "patient_id_anon" in train_df.columns else "trustii_id"
patient_ids = train_df[id_col]


def build_ranking_comparison(X, y, event, time, ml_scores, formula_scores, patient_ids):
    ml_scores = np.asarray(ml_scores)
    formula_scores = np.asarray(formula_scores)
    event_name, time_name = y.dtype.names

    ml_rank = pd.Series(ml_scores, index=X.index).rank(ascending=False, method="min")
    formula_rank = pd.Series(formula_scores, index=X.index).rank(ascending=False, method="min")

    return pd.DataFrame({
        "patient_id": patient_ids.reindex(X.index).to_numpy(),
        "ml_score": ml_scores,
        "ml_rank": ml_rank.to_numpy(),
        "formula_score": formula_scores,
        "formula_rank": formula_rank.to_numpy(),
        "rank_gap": (formula_rank - ml_rank).to_numpy(),
        "event": y[event_name],
        "time": y[time_name],
    })


ranking_fib4 = pd.concat([
    build_ranking_comparison(X_hep, y_hep, hep_event, hep_time, hep_blend_oof, fib4_apri_hep["fib4_score"], patient_ids).assign(endpoint="hepatic"),
    build_ranking_comparison(X_death, y_death, death_event, death_time, death_blend_oof, fib4_apri_death["fib4_score"], patient_ids).assign(endpoint="death"),
], ignore_index=True)

ranking_apri = pd.concat([
    build_ranking_comparison(X_hep, y_hep, hep_event, hep_time, hep_blend_oof, fib4_apri_hep["apri_score"], patient_ids).assign(endpoint="hepatic"),
    build_ranking_comparison(X_death, y_death, death_event, death_time, death_blend_oof, fib4_apri_death["apri_score"], patient_ids).assign(endpoint="death"),
], ignore_index=True)

ranking_fib4 = ranking_fib4.reindex(ranking_fib4["rank_gap"].abs().sort_values(ascending=False).index).reset_index(drop=True)
ranking_apri = ranking_apri.reindex(ranking_apri["rank_gap"].abs().sort_values(ascending=False).index).reset_index(drop=True)

ranking_fib4_path = OUTPUTS_DIR / "ranking_comparison_fib4.csv"
ranking_apri_path = OUTPUTS_DIR / "ranking_comparison_apri.csv"

ranking_fib4.to_csv(ranking_fib4_path, index=False)
ranking_apri.to_csv(ranking_apri_path, index=False)

print(f"Wrote {ranking_fib4_path} ({len(ranking_fib4)} rows)")
print(f"Wrote {ranking_apri_path} ({len(ranking_apri)} rows)")

ranking_fib4.head(10)


## Summary

For each endpoint, which method wins on the honest CV comparison, and
whether the 95% CI on the gap (blend - formula) excludes zero.

In [ ]:
def verdict(lo, hi):
    if lo > 0:
        return "blend significantly better"
    if hi < 0:
        return "formula significantly better"
    return "not statistically significant"


summary_rows = []
for endpoint, blend_cv, fib4_cv, apri_cv, fib4_lo, fib4_hi, apri_lo, apri_hi in [
    ("hepatic", hep_blend_cv, hep_fib4_cv, hep_apri_cv, hep_fib4_lo, hep_fib4_hi, hep_apri_lo, hep_apri_hi),
    ("death", death_blend_cv, death_fib4_cv, death_apri_cv, death_fib4_lo, death_fib4_hi, death_apri_lo, death_apri_hi),
]:
    scores = {"Blend": blend_cv[0], "FIB-4": fib4_cv[0], "APRI": apri_cv[0]}
    winner = max(scores, key=scores.get)
    summary_rows.append({
        "endpoint": endpoint,
        "winner (mean CV c-index)": winner,
        "blend_mean": round(blend_cv[0], 4),
        "fib4_mean": round(fib4_cv[0], 4),
        "apri_mean": round(apri_cv[0], 4),
        "blend_vs_fib4_95CI": f"[{fib4_lo:.4f}, {fib4_hi:.4f}] -- {verdict(fib4_lo, fib4_hi)}",
        "blend_vs_apri_95CI": f"[{apri_lo:.4f}, {apri_hi:.4f}] -- {verdict(apri_lo, apri_hi)}",
    })

summary_df = pd.DataFrame(summary_rows)
pd.set_option("display.max_colwidth", None)
summary_df
